In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"

DB_PATH = PROJECT_ROOT / "data" / "ecommerce.db"

print(DB_PATH)

/Users/ayushgiri/ecommerce-customer-revenue-analytics/data/ecommerce.db


In [3]:
conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

print("Database created successfully.")

Database created successfully.


In [4]:
customers = pd.read_csv(RAW_DATA / "olist_customers_dataset.csv")

orders = pd.read_csv(RAW_DATA / "olist_orders_dataset.csv")

order_items = pd.read_csv(RAW_DATA / "olist_order_items_dataset.csv")

payments = pd.read_csv(RAW_DATA / "olist_order_payments_dataset.csv")

products = pd.read_csv(RAW_DATA / "olist_products_dataset.csv")

reviews = pd.read_csv(RAW_DATA / "olist_order_reviews_dataset.csv")

sellers = pd.read_csv(RAW_DATA / "olist_sellers_dataset.csv")

translation = pd.read_csv(
    RAW_DATA / "product_category_name_translation.csv"
)

In [5]:
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

order_items.to_sql(
    "order_items",
    conn,
    if_exists="replace",
    index=False
)

payments.to_sql(
    "payments",
    conn,
    if_exists="replace",
    index=False
)

products.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

reviews.to_sql(
    "reviews",
    conn,
    if_exists="replace",
    index=False
)

sellers.to_sql(
    "sellers",
    conn,
    if_exists="replace",
    index=False
)

translation.to_sql(
    "category_translation",
    conn,
    if_exists="replace",
    index=False
)

71

In [6]:
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

tables

,name
0,customers
1,orders
2,order_items
3,payments
4,products
5,reviews
6,sellers
7,category_translation


In [7]:
for table in tables["name"]:

    count = pd.read_sql(
        f"""
        SELECT COUNT(*) AS rows
        FROM {table}
        """,
        conn
    )

    print(table, count.iloc[0, 0])

customers 99441
orders 99441
order_items 112650
payments 103886
products 32951
reviews 99224
sellers 3095
category_translation 71


In [8]:
query = """
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

pd.read_sql(query, conn)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [9]:
conn.commit()
conn.close()